# 🚀 Pandas vs Polars Benchmark pe Titanic

In [12]:
import pandas as pd

# URL către datasetul Titanic (CSV public)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

# Setăm opțiuni de afișare pentru a vedea mai bine tabelele în notebook (util: debugging/inspecție)
pd.set_option("display.max_columns", 100)  # arată până la 100 de coloane
pd.set_option("display.width", 100)        # lățimea maximă a liniei

# === Citire cu Pandas direct din URL ===
pd_df = pd.read_csv(url)
pd_df.head()



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
import polars as pl
import requests
from io import BytesIO

# === Citire cu Polars din URL (folosim requests + BytesIO) ===
content = requests.get(url).content
pl_df = pl.read_csv(BytesIO(content))
pl_df.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


## 1. Test citire CSV

In [14]:
start = time.time()
pd.read_csv(url)
print("⏱ Pandas read:", time.time() - start)

start = time.time()
pl.read_csv(url)
print("⏱ Polars read:", time.time() - start)

⏱ Pandas read: 0.05988287925720215
⏱ Polars read: 0.05633950233459473


## 2. GroupBy + agregare (rata de supraviețuire pe Pclass și Sex)

In [28]:
start = time.time()
pd_result = (
    pd_df.groupby(["Pclass", "Sex"])["Survived"].mean().reset_index()
)
# print(pd_result)
print("⏱ Pandas groupby:", time.time() - start)

start = time.time()
pl_result = (
    pl_df.group_by(["Pclass", "Sex"]).agg(pl.col("Survived").mean())
)
# print(pl_result)
print("⏱ Polars groupby:", time.time() - start)

⏱ Pandas groupby: 0.004935264587402344
⏱ Polars groupby: 0.001386404037475586


## 3. Sortare după tarif (Fare)

In [26]:
start = time.time()
pd_sorted = pd_df.sort_values("Fare", ascending=False)
print("⏱ Pandas sort:", time.time() - start)

start = time.time()
pl_sorted = pl_df.sort("Fare", descending=True)
print("⏱ Polars sort:", time.time() - start)

⏱ Pandas sort: 0.012630939483642578
⏱ Polars sort: 0.020553112030029297


## 4. Filtrare copii (<12 ani)

In [30]:
start = time.time()
pd_children = pd_df[pd_df["Age"] < 12]
print("⏱ Pandas filter:", time.time() - start)

start = time.time()
pl_children = pl_df.filter(pl.col("Age") < 12)
print("⏱ Polars filter:", time.time() - start)

⏱ Pandas filter: 0.0013914108276367188
⏱ Polars filter: 0.019047975540161133


## 5. Benchmark cu timeit (1000 rulări)

In [37]:
print("⏱ Pandas groupby (1000 runs):", timeit.timeit(
    lambda: pd_df.groupby(["Pclass", "Sex"])["Survived"].mean(),
    number=1000
))

print("⏱ Polars groupby (1000 runs):", timeit.timeit(
    lambda: pl_df.group_by(["Pclass", "Sex"]).agg(pl.col("Survived").mean()),
    number=1000
))

⏱ Pandas groupby (1000 runs): 1.0429381839999223
⏱ Polars groupby (1000 runs): 0.6293214980000812
